In [1]:
################### Emulation with a for loop ########

from datetime import datetime, timedelta
import os
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import requests
import sys
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiPoint
from shapely.geometry import Point
import fiona
import contextily as ctx
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
import simplekml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from pyproj import Proj, transform
import ast
from pyproj import Transformer
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.warp import calculate_default_transform, reproject
from rasterio.enums import Resampling

from scipy.ndimage import binary_fill_holes
from scipy.ndimage import convolve
from scipy.ndimage import distance_transform_edt
from affine import Affine
from scipy.spatial import KDTree
from rasterio.mask import mask
from rasterio.features import shapes
from shapely.geometry import shape
from simplekml import Kml
from osgeo import gdal
from rasterio.transform import rowcol

In [2]:
def count_non_nan_values(array):
        """Counts non-NaN values in an array while ignoring non-numeric elements."""
        # Filter to keep only numeric data (e.g., ints and floats)
        numeric_values = [x for x in array if isinstance(x, (int, float))] 
        # Convert to NumPy array and count non-NaN values
        return np.sum(~np.isnan(np.array(numeric_values, dtype=float)))
    
def data_prep(path, file, cutoff=0, direction=0, cutlon=0, sensor = 1):
    ####### filter the data with 015042 index ######
    df = pd.read_csv(path + file)
    df['ID'] = range(0, len(df))  # Generates IDs starting from 1
    
    if sensor == 0:
        training_data = df[['lat', 'lon', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'ndmi', 'ndvi', 'savi', 'descriptio']].to_numpy()
    else:
        training_data = df[['lat', 'lon', 'SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'ndmi', 'ndvi', 'savi', 'descriptio']].to_numpy()
    
    predictor = []
    target = []
    coord = []
    # Loop through each (1585, 10) array in training_data
    print('training data is:')
    #print(training_data)
    print(training_data.shape)
    #for array in training_data:
        # Get rows where the target is not 'TZ'
    mask = training_data[:, 11] != 'MZ'
    
    predictor.append(training_data[mask, 2:11])  # First 9 columns
    target.append(training_data[mask, 11])       # Target column
    coord.append(training_data[mask, :2])
    
    # Convert to NumPy arrays if desired
    predictor = np.array(predictor)
    target = np.array(target)
    coord = np.array(coord)
    
    X = predictor.reshape(-1, 9)  # Shape: (5 * 1585, 9)
    y = target.reshape(-1)        # Shape: (5 * 1585,)
    coord = coord.reshape(-1, 2)
    ### cleanse the data by removing nan ####
    valid_indices = []

    # Loop through each row and check for NaN in the first column
    for i in range(X.shape[0]):
        if not np.isnan(X[i, 0]):
            valid_indices.append(i)
    
    # Create new X and y with only valid rows
    X = X[valid_indices]
    y = y[valid_indices]
    coord = coord[valid_indices]

    print('the size of dataset is:')
    print(X.shape[0])

    if cutoff:
        if direction > 0: ### keep the eastern side
            masklon = coord[:, 1] > cutlon
        else:
            masklon = coord[:, 1] <= cutlon
    
        X = X[masklon]
        y = y[masklon]
        coord = coord[masklon]
    
        print('the size of lon filtered dataset is:')
        print(X.shape[0])
        #print(X)
    
    return X, y, coord

In [3]:
def model_train_test(X, y):
    # Initialize the RandomForestClassifier
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    
    # Fit the model to the reshaped data
    model.fit(X, y)
    
    # After training, the model can be used for predictions or evaluation
    y_pred = model.predict(X)
    
    # Calculate accuracy by comparing the predictions with the true labels (y)
    accuracy = model.score(X, y)
    
    print(f"Training and testing using the whole dataset, accuracy is: {accuracy * 100:.2f}%")

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    
    # Optionally, evaluate the model on the test set
    score = model.score(X_test, y_test)
    print(f"Testing using 20% of the whole dataset, accuracy is: {score*100:.2f}%")
    
    y_pred = model.predict(X_test)

    print("the number of elements for the prediction:")
    print(y_pred.shape[0])
    cm = confusion_matrix(y_test, y_pred, labels=['VZ', 'TZ', 'WZ'])
    
    # Plot the confusion matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['VZ', 'TZ', 'WZ'])
    disp.plot(cmap='Blues', values_format='.0f')
    
    return accuracy

In [4]:
def model_across_years(X, y, X_test, y_test, coord_test):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X, y)
    score = model.score(X_test, y_test)
    print(f"Testing across years, accuracy is: {score*100:.2f}%")
    y_pred = model.predict(X_test)
    print(y_pred)
    cm = confusion_matrix(y_test, y_pred, labels=['VZ', 'TZ', 'WZ'])
    
    # Plot the confusion matrix
    #disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['VZ', 'TZ', 'WZ'])
    #disp.plot(cmap='Blues', values_format='.0f')

    incorrect_indices = y_test != y_pred  # Boolean array where predictions are wrong
    misclassified_data = coord_test[incorrect_indices]  # Coordinates of misclassified samples
    misclassified_y_true = y_test[incorrect_indices]  # True values of misclassified samples
    misclassified_y_pred = y_pred[incorrect_indices]  # Predicted values of misclassified samples
    
    # Output misclassified elements
    
    # for i in range(len(misclassified_data)):
    #     print(f"Coord: {misclassified_data[i]}, Predicted: {misclassified_y_pred[i]}, True: {misclassified_y_true[i]}")
    return y_pred

In [5]:
def extract_coordinates(row):
    try:
        # Parse the JSON-like string to a dictionary
        data = ast.literal_eval(row)
        coordinates = data['coordinates']
        lon = round(coordinates[0], 10)
        lat = round(coordinates[1], 10)
        return coordinates[0], coordinates[1]  # lon, lat
    except (ValueError, KeyError):
        return None, None  # Handle parsing errors gracefully

In [6]:
def define_consistency_dataset(coord_cal1, y_cal1, coord_cal2, y_cal2):
    # Convert to pandas DataFrame for easier comparison
    # Convert to pandas DataFrame for easier comparison
    df1 = pd.DataFrame({'lat': coord_cal1[:,0], 'lon': coord_cal1[:,1], 'y': y_cal1})
    df2 = pd.DataFrame({'lat': coord_cal2[:,0], 'lon': coord_cal2[:,1], 'y': y_cal2})

    # Find common rows based on lat, lon, and y
    merged_df = df1.merge(df2, on=['lat', 'lon', 'y'], how='inner', indicator=True)

    ##print("Matching indices:", matching_indices)

    # Get indices from df1
    matching_indices_df1 = df1.index[df1[['lat', 'lon', 'y']].apply(tuple, axis=1).isin(
        merged_df[['lat', 'lon', 'y']].apply(tuple, axis=1)
    )].tolist()

    # Get indices from df2
    matching_indices_df2 = df2.index[df2[['lat', 'lon', 'y']].apply(tuple, axis=1).isin(
        merged_df[['lat', 'lon', 'y']].apply(tuple, axis=1)
    )].tolist()

    # Output results
    #print("Common elements and their indices in df1:")
    #print(df1.loc[matching_indices_df1])

    #print("\nIndices of the same elements in df2:")
    #print(df2.loc[matching_indices_df2])

    common_df1 = df1.loc[matching_indices_df1].to_numpy()
    common_df2 = df2.loc[matching_indices_df2].to_numpy()
    print('common indices are:')
    print(y_cal1[matching_indices_df1])
    return common_df1, matching_indices_df1

In [7]:
def calculate_accuracy_target_year(common_df1, coord_pred, y_pred):
    ################## Evaluate the prediction accuracy for the target year #####################
    # Step 1: Find matching coordinates between coord_pred and common_df1
    common_coords = common_df1[:, :2]  # Extract lat, lon from common_df1

    # Find indices where coord_pred exists in common_coords
    matching_indices = [i for i, coord in enumerate(coord_pred) if any(np.all(coord == common_coords[j]) for j in range(common_coords.shape[0]))]

    # Step 2: Get y values from common_df1 based on matching indices
    y_true = np.array([common_df1[np.where((common_coords == coord_pred[i]).all(axis=1))[0][0], 2] for i in matching_indices])

    # Step 3: Get subset of y_pred for matching coordinates
    y_pred_subset = y_pred[matching_indices]

    # Step 4: Compute overall accuracy (Mean Absolute Error or % Accuracy)
    accuracy = np.mean(y_pred_subset == y_true) * 100 

    # Output results

    print(f"Accuracy of the predicted values compared to the consistency points between two cal years: {accuracy:.1f}%")
    return np.round(accuracy,1)

In [10]:
ifcombine = 1
sensorshift1 = 1
sensorshift2 = 1

calyear1 = 1985
calyear2 = 1994
calyear3 = 1999
calyear4 = 2003

threshold = 0.65 ## using this to control the reclassification to TZ
folder = 'L5/'

X_cal1, y_cal1, coord_cal1 = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/' + str(calyear1) + '/', 'median.csv', 0, 0, 0, sensorshift1) ## mixing 2022 Mar and 2023 Feb
X_cal1 = X_cal1.astype(float)

X_cal2, y_cal2, coord_cal2 = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/' + str(calyear2) + '/', 'median.csv', 0, 0, 0, sensorshift1) ## mixing 2022 Mar and 2023 Feb
X_cal2 = X_cal2.astype(float)

X_cal3, y_cal3, coord_cal3 = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/' + str(calyear3) + '/', 'median.csv', 0, 0, 0, sensorshift2) ## mixing 2022 Mar and 2023 Feb
X_cal3 = X_cal3.astype(float)

X_cal4, y_cal4, coord_cal4 = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/' + str(calyear4) + '/', 'median.csv', 0, 0, 0, sensorshift2) ## mixing 2022 Mar and 2023 Feb
X_cal4 = X_cal4.astype(float)

training data is:
(1579, 12)
the size of dataset is:
1439
training data is:
(1573, 12)
the size of dataset is:
1442
training data is:
(1577, 12)
the size of dataset is:
1444
training data is:
(1581, 12)
the size of dataset is:
1448


In [11]:
################ FOR LOOP FOR EVERYTHING #####################################################

for predyear in range(1984, 2003):  # Loop from 2014 to 2018
    if predyear == 1985 or predyear == 1994 or predyear == 1999:
        continue
    print('*************************************************')
    print('the year is')
    print(predyear)
    inclusive = 1 if calyear1 <= predyear <= calyear2 else 0
    sensorshift = 0 if predyear >= 2014 else 1
    
    X_pred, y_pred_fake, coord_pred = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/Emulation/' + folder, 'points_' + str(predyear) + '.csv', 0, 0, 0, sensorshift) ## mixing 2022 Mar and 2023 Feb
    X_pred = X_pred.astype(float)
    rows_with_nan = np.any(np.isnan(X_pred), axis=1)
    indices_with_nan = np.where(rows_with_nan)[0]
    print("indices with nan are:")
    print(indices_with_nan)

    ###################### Define consistency dataset ###################
   
    if 1985 < predyear < 1994:
        df1, common_indices = define_consistency_dataset(coord_cal1, y_cal1, coord_cal2, y_cal2) ## this is to make the training data
    elif 1994 < predyear < 1999:
        df1, common_indices = define_consistency_dataset(coord_cal2, y_cal2, coord_cal3, y_cal3) 
    elif 1999 < predyear < 2003:
        df1, common_indices = define_consistency_dataset(coord_cal3, y_cal3, coord_cal4, y_cal4) 
        
    #X_csd = X_cal1[common_indices] ## consistency points
    y_csd = y_cal1[common_indices]
    coord_csd = coord_cal1[common_indices] ## find the common y and coord for the two GE years
    
    common_mask = (coord_csd[:, None] == coord_pred).all(-1).any(-1)
    indices_csd = np.where(common_mask)[0]  # Indices in coord_csd

    common_mask_pred = (coord_pred[:, None] == coord_csd).all(-1).any(-1)
    indices_pred = np.where(common_mask_pred)[0]  # Indices in coord_pred

    y_csd = y_csd[indices_csd] ## using the label from the common GE years
    X_csd = X_pred[indices_pred] ## using the X for the target year
    
    print('shape check:')
    print(y_csd.shape)
    print(X_csd.shape)
    ############ Temporal transfer learning and accuracy assessment ######
    ############ Using the consistency points ! ##########################
    if ifcombine == 1:
        X_cal = np.vstack((X_cal1, X_cal2, X_csd))
        y_cal = np.vstack((y_cal1.reshape(-1, 1), y_cal2.reshape(-1, 1), y_csd.reshape(-1, 1)))
        y_pred = model_across_years(X_cal, y_cal, X_pred, y_pred_fake, coord_pred)
    else: ## if it is 0, then only use the first one. 
        X_cal = np.vstack((X_cal1, X_csd))
        y_cal = np.vstack((y_cal1.reshape(-1, 1), y_csd.reshape(-1, 1)))
        y_pred = model_across_years(X_cal, y_cal, X_pred, y_pred_fake, coord_pred)
    
    print(f"Processing for predyear: {predyear}")
    if inclusive: ## if the target year is in between of the training cal years
        acc = calculate_accuracy_target_year(df1, coord_pred, y_pred)
    else:

        if 1985 < predyear < 1994:
             df1, common_indices = define_consistency_dataset(coord_cal1, y_cal1, coord_cal2, y_cal2) ## between 2018 and 2013
        elif 1994 < predyear < 1999:
             df1, common_indices = define_consistency_dataset(coord_cal2, y_cal2, coord_cal3, y_cal3)
        elif 1999 < predyear < 2003:
             df1, common_indices = define_consistency_dataset(coord_cal3, y_cal3, coord_cal4, y_cal4)
                
        acc = calculate_accuracy_target_year(df1, coord_pred, y_pred)
    
    ############# Using the predicted y and X train the model and map the region ########

    path = "/Users/bz294/Documents/WhiteZone/Points/KML/Emulation/" + folder
    file = 'full_' + str(predyear) + '.csv'
    region_df = pd.read_csv(path + file)

    ############ data prep #############

    region_df['lon'], region_df['lat'] = zip(*region_df['.geo'].apply(extract_coordinates))
    coordinates = region_df[['lon', 'lat']].to_numpy()
    feature = region_df.iloc[:, 1:12].drop(region_df.columns[[7, 10]], axis=1).to_numpy()

    rows_with_nan = np.any(np.isnan(feature), axis=1)

    # Get indices of rows with NaN
    indices_with_nan = np.where(rows_with_nan)[0]
    print("indices with nan are:")
    print(indices_with_nan)

    feature_cleaned = np.delete(feature, indices_with_nan, axis=0)
    coordinates_cleaned = np.delete(coordinates, indices_with_nan, axis=0)

    transformer = Transformer.from_crs("EPSG:4326", "EPSG:32617", always_xy=True)

    # Transform the coordinates
    coord_utm = np.array([
        transformer.transform(lon, lat) for lon, lat in coordinates_cleaned
    ])
    
    ################## Model training ###########################
    threshold = 0.65
    model = RandomForestClassifier(n_estimators=100, random_state=42)

    # Fit the model to the reshaped data
    model.fit(X_pred, y_pred)

    # After training, the model can be used for predictions or evaluation
    y_region_proba = model.predict_proba(feature_cleaned)
    y_region = model.predict(feature_cleaned)
    y_region_change = y_region.copy()

    final_class_probs = np.max(y_region_proba, axis=1)
    # Find indices where the predicted probability is less than the threshold
    low_confidence_indices = final_class_probs < threshold

    # Update y_region to 'TZ' where confidence is low
    y_region_change[low_confidence_indices] = 'TZ'

    class_mapping = {'VZ': 1, 'WZ': 2, 'TZ': 3}

    # Convert y_region_change from string to numerical values using the mapping
    y_region_change_values = np.vectorize(class_mapping.get)(y_region_change)
    
    ############ Output the shapefile ##############
    
    outputpath = '/Users/bz294/Documents/WhiteZone/Points/KML/Emulation/Output/'
    filename = outputpath + 'output_points_' + str(predyear) + '.shp'
    geometry = [Point(xy) for xy in coord_utm]

    # Create a GeoDataFrame
    gdf = gpd.GeoDataFrame({'y_region': y_region_change_values}, geometry=geometry)

    # Set the CRS to UTM 17N (EPSG:32617)
    gdf.set_crs('EPSG:32617', inplace=True)

    # Save to a shapefile
    output_shapefile = filename  # Define your output shapefile path
    gdf.to_file(output_shapefile)

    print(f'Shapefile saved to: {output_shapefile}')
    
    ############## Producing rasters #############################
    
    shapefile_path = filename
    gdf = gpd.read_file(shapefile_path)
    region_mapping = {'VZ': 1, 'WZ': 2, 'TZ': 3}
    #gdf['y_region_mapped'] = gdf['y_region'].map(region_mapping)

    gdf['y_region_mapped'] = gdf['y_region']
    polygon_path = "/Users/bz294/Documents/WhiteZone/Points/Shapefiles/SEregion_reprojected.shp" ## it is the utm projected
    polygon_gdf = gpd.read_file(polygon_path)
    geometries = [feature["geometry"] for feature in polygon_gdf.__geo_interface__["features"]]

    xmin, ymin, xmax, ymax = gdf.total_bounds
    pixel_size = 30  # Define the pixel size
    cols = int((xmax - xmin) / pixel_size)
    rows = int((ymax - ymin) / pixel_size)
    transform = from_origin(xmin, ymax, pixel_size, pixel_size)
    # Get coordinates and y_region values
    coords = np.array([(point.x, point.y) for point in gdf.geometry])
    values = gdf['y_region_mapped'].values
    ##np.save("coords.npy", np.array(coords))  # Save as .npy file
    
    ############ Build a KDTree for nearest-neighbor search #########
    tree = KDTree(coords)

    # Create an empty raster
    raster = np.empty((rows, cols), dtype=np.float32)
    raster.fill(np.nan)

    # Iterate over the raster grid
    x_coords = np.linspace(xmin + pixel_size / 2, xmax - pixel_size / 2, cols)
    y_coords = np.linspace(ymax - pixel_size / 2, ymin + pixel_size / 2, rows)
    for i, y in enumerate(y_coords):
        for j, x in enumerate(x_coords):
            # Find the nearest point
            _, idx = tree.query((x, y))
            raster[i, j] = values[idx]

    # Save the raster
    output_raster_path = "raster_2022.tif"
    with rasterio.open(
        output_raster_path,
        'w',
        driver='GTiff',
        height=rows,
        width=cols,
        count=1,
        dtype=raster.dtype,
        crs=gdf.crs.to_string(),
        transform=transform
    ) as dst:
        dst.write(raster, 1)

    print(f"Raster saved to {output_raster_path}")
    
    ######### Read it back and mask the outside areas #########
    with rasterio.open(output_raster_path) as src:
        out_image, out_transform = mask(src, geometries, crop=False) ## mask out based on the SE region shapefile
        out_meta = src.meta

    # Update metadata to account for changes
    out_meta.update({"height": out_image.shape[1], "width": out_image.shape[2], "transform": out_transform})

    path = '/Volumes/Malonelab/Research/Whitezone/Result/Emulation/'
    if ifcombine == 0:
        output_raster_file = 'raster_' + str(predyear) + '_' + str(calyear1) + 'threshold.tif'
    else:
        output_raster_file = 'raster_' + str(predyear) + '_' + str(calyear1) + '_' + str(calyear2) + 'threshold.tif'
    #output_raster_path = 'raster_' + str(predyear) + '_2018.tif'
    # Save the masked raster
    with rasterio.open(path + output_raster_file, "w", **out_meta) as dst:
        dst.write(out_image)

    print(f"Masked raster saved to {path + output_raster_file}")

*************************************************
the year is
1984


FileNotFoundError: [Errno 2] No such file or directory: '/Users/bz294/Documents/WhiteZone/Points/KML/Emulation/L5/points_1984.csv'

In [ ]:
y_csd = y_cal1[np.array(common_indices)]

In [ ]:
y_cal1.shape

In [ ]:
type(common_indices)


In [ ]:
print(np.array(common_indices).max(), len(y_cal1))